# Volatility threshold regimes

This backtest uses Yang–Zhang volatility, rolling thresholds, transaction costs and walk-forward validation. The default configuration uses URTH.

Daily OHLCV comes from Yahoo Finance via yfinance. Open, high, low and close are adjusted for splits and dividends on the same basis. Outputs are saved to results/volatility-threshold-regimes/.

In [ ]:
"""
================================================================================
THRESHOLD REGIME TRADING STRATEGY - VERSION 2.1
================================================================================


IMPROVEMENTS FROM V1.0:
-----------------------
This version addresses the critical failures identified in the V1.0 backtest:

0. CRITICAL FIX V2.1 - EXECUTION LOGIC:
   - V2.0 BUG: Daily rebalancing to maintain fixed $100k notional caused
     spurious trades (491 trades in 504 days = trading every day!)
   - This was NOT regime trading - it was price-drift rebalancing
   - V2.1 FIX: Trade ONLY when SIGNAL changes (regime transition)
   - Let position "drift" with market when regime unchanged
   - Expected trades: ~10-30/year (actual regime changes) vs ~500/year

1. INTRA-FOLD PARAMETER OPTIMIZATION:
   - V1.0 used fixed parameters across all walk-forward folds
   - V2.0 optimizes hyperparameters (vol_lookback, quantile_window, regime_quantile)
     within each training window using grid search + differential evolution
   - This evaluates the CONCEPT's robustness, not just one parameter set

2. SIGNAL PERSISTENCE FILTER:
   - V1.0 had 1015 trades in ~5 years (excessive whipsaws, high costs)
   - V2.0 requires regime to persist for N days before executing
   - Reduces noise-induced trading and transaction costs
   - Configurable persistence_days parameter (default: 3 days)

3. FIXED DSR CALCULATION:
   - V1.0 showed DSR=0 which contradicted PSR=1.0
   - V2.0 implements corrected Bailey-Lopez de Prado formula
   - Proper adjustment for number of trials in walk-forward

4. COMPREHENSIVE VISUALIZATIONS:
   - Equity curve vs benchmark with drawdown overlay
   - Per-fold IS vs OOS Sharpe comparison
   - Regime overlay on price chart
   - Rolling IC (Information Coefficient) analysis
   - Monthly returns heatmap
   - Robustness metrics radar chart

5. ZERO-SHARPE FOLD DIAGNOSTICS:
   - Tracks days with no signal changes per fold
   - Identifies periods where regime threshold wasn't crossed
   - Reports fold-level trade statistics

6. IMPROVED WALK-FORWARD STRUCTURE:
   - Train -> Optimize -> Validate -> Test pipeline
   - Expanding window with proper temporal isolation
   - Records optimal parameters per fold for stability analysis

STRATEGY OVERVIEW:
-----------------
Uses volatility-based threshold regime detection to dynamically allocate
between a global equity ETF (URTH) and cash. High-volatility regimes 
historically show poor risk-adjusted returns; low-volatility regimes are better.

METHODOLOGY:
-----------
1. REGIME DETECTION (Threshold Model with Persistence):
   - Calculate Yang-Zhang volatility using OHLC data
   - Use ROLLING quantiles (not global) to define regime thresholds
   - Apply persistence filter: regime must hold N days before signal change
   - Regime 0 (Low Vol): Full equity exposure
   - Regime 1 (High Vol): Reduce/eliminate exposure

2. TEMPORAL INTEGRITY (Zero Look-Ahead Bias):
   - Signal generated at Close[T-1] using only data up to T-1
   - Order executed at Open[T] the next trading day
   - Position sized using Close[T-1] prices and portfolio value
   - Rolling quantile windows use closed='left' to exclude current observation
   - All feature engineering explicitly uses shift(1)

3. WALK-FORWARD VALIDATION WITH INTRA-FOLD OPTIMIZATION:
   - Expanding window approach with periodic model refitting
   - For each fold: optimize parameters on Train, evaluate on Validate
   - Out-of-sample performance tracking for robustness assessment
   - Parameter stability analysis across folds

4. HYBRID P&L CALCULATION:
   - Execution days: Use Open[T]/Open[T-1] - 1 for return calculation
   - Holding days: Use Close[T]/Close[T-1] - 1 for return calculation
   - Properly accounts for execution slippage and overnight gaps

5. POSITION SIZING:
   - RESEARCH MODE (COMPOUNDING_MODE=False): Fixed notional on initial capital
   - PRODUCTION MODE (COMPOUNDING_MODE=True): Geometric compounding
   - Cash reserve tracks unrealized gains/losses in research mode

6. TRANSACTION COSTS (IBKR-like):
   - Commission: 0.0035 per share, min $0.35, max 1% of trade value
   - Spread: 0.01% per side (liquid ETFs)
   - Market Impact: 0.02% per side for larger trades
   - Margin Interest: Fed Funds + 1.5% for short positions (if any)

ROBUSTNESS METRICS:
------------------
- Deflated Sharpe Ratio (DSR): Adjusts for multiple testing bias (FIXED)
- Probabilistic Sharpe Ratio (PSR): Statistical significance of Sharpe
- Walk-Forward Efficiency (WFE): OOS/IS performance ratio
- Hurst Exponent: Mean-reversion vs trend persistence
- CDaR (Conditional Drawdown at Risk): Tail risk measure
- PSI (Parameter Stability Index): Sensitivity analysis across folds
- Rolling IC: Predictive power consistency
- Global Robustness Index (GRI): Combined normalized score

DATA SOURCE:
-----------
Daily OHLCV from Yahoo Finance via yfinance

BENCHMARK: URTH (iShares MSCI World ETF)

================================================================================
"""

from pathlib import Path

import numpy as np
import pandas as pd
from typing import Dict, List, Optional, Tuple, Any, Callable
from dataclasses import dataclass, field
from datetime import datetime, timedelta
import warnings
from scipy import stats
from scipy.optimize import minimize, differential_evolution
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch
import seaborn as sns
from itertools import product
import json

from _market_data import download_ohlcv

warnings.filterwarnings('ignore')

# ==============================================================================
# CONFIGURATION CLASS
# ==============================================================================
# Centralized configuration to avoid hardcoded parameters throughout the code.
# All parameters are explicitly documented and can be modified from a single location.
#
# KEY IMPROVEMENT V2.0:
# - Added persistence_days parameter for signal filtering
# - Added parameter optimization ranges for grid search
# - Added hyperparameter optimization configuration
# ==============================================================================

@dataclass
class StrategyConfig:
    """
    Master configuration class for the Threshold Regime Trading Strategy V2.0.
    
    DESIGN PHILOSOPHY:
    - All parameters in one place for easy modification
    - No magic numbers scattered throughout the codebase
    - Clear documentation of each parameter's purpose
    - Sensible defaults based on academic literature and practice
    
    PARAMETER CATEGORIES:
    1. Data Parameters: ticker symbols and dates
    2. Regime Detection: Threshold model hyperparameters
    3. Signal Filtering: Persistence requirements (NEW in V2.0)
    4. Position Sizing: Capital management and leverage
    5. Transaction Costs: Realistic IBKR-like fee structure
    6. Walk-Forward: Validation window sizes and frequencies
    7. Hyperparameter Optimization: Grid search ranges (NEW in V2.0)
    8. Robustness: Statistical testing thresholds
    9. Visualization: Plot settings
    """
    
    # -------------------------------------------------------------------------
    # DATA PARAMETERS
    # -------------------------------------------------------------------------
    # Primary trading asset - Global equity exposure via MSCI World ETF
    ticker: str = "URTH"
    
    # Benchmark for performance comparison
    # Using URTH as both asset and benchmark creates a "beat the market" scenario
    benchmark_ticker: str = "URTH"
    
    # Backtest period: 5 years ending 2018-12-31
    # NOTE: We request extra data before start_date for feature warm-up
    # The actual backtest will begin after warm-up period completes
    start_date: str = "2014-01-01"  # Actual trading starts after warm-up
    end_date: str = "2018-12-31"
    
    # Additional buffer before start_date for feature calculation
    # Must be >= max(quantile_window) + max(vol_lookback) + safety margin
    # With max quantile_window=504 and max vol_lookback=63, need ~600 days
    warmup_buffer_days: int = 700  # ~2.8 years of trading days for safety
    
    # -------------------------------------------------------------------------
    # REGIME DETECTION PARAMETERS (THRESHOLD MODEL)
    # -------------------------------------------------------------------------
    # These are DEFAULT values; will be optimized during walk-forward
    
    # Volatility calculation window (Yang-Zhang estimator)
    # 20 days ~ 1 month of trading, balances responsiveness vs noise
    vol_lookback: int = 20
    
    # Rolling window for calculating quantile thresholds
    # 252 days = 1 trading year, provides stable threshold estimation
    # CRITICAL: Uses closed='left' to prevent look-ahead bias
    quantile_window: int = 252
    
    # Number of regimes (binary: low-vol vs high-vol)
    n_regimes: int = 2
    
    # Quantile threshold for regime classification
    # 0.5 = median split -> equal probability of each regime ex-ante
    regime_quantile: float = 0.5
    
    # Yang-Zhang volatility estimator parameters
    # These weights optimize the estimator's efficiency
    # Reference: Yang & Zhang (2000) "Drift Independent Volatility Estimation"
    yz_k_numerator: float = 0.34
    yz_k_denominator_constant: float = 1.34
    yz_annualization_factor: int = 252
    
    # -------------------------------------------------------------------------
    # SIGNAL PERSISTENCE FILTER (NEW IN V2.0)
    # -------------------------------------------------------------------------
    # PROBLEM ADDRESSED:
    # V1.0 had 1015 trades in ~5 years, causing excessive whipsaws and costs
    # This was due to noisy regime transitions triggering immediate trades
    #
    # SOLUTION:
    # Require regime to persist for N consecutive days before changing signal
    # This filters out short-term noise and reduces unnecessary trading
    
    # Minimum days regime must persist before signal change
    # 3 days is a reasonable starting point; can be optimized
    persistence_days: int = 3
    
    # Enable/disable persistence filter
    # Set to False to compare with V1.0 behavior
    use_persistence_filter: bool = True
    
    # -------------------------------------------------------------------------
    # POSITION SIZING PARAMETERS
    # -------------------------------------------------------------------------
    # CRITICAL: These parameters control how positions are sized
    # Read the docstring at the top of this file for detailed explanation
    
    # Initial capital for backtest
    initial_capital: float = 100000.0
    
    # COMPOUNDING_MODE = False (RESEARCH MODE):
    #   - Positions sized on INITIAL_CAPITAL (fixed notional)
    #   - Returns reflect pure signal alpha
    #   - Cash reserve tracks unrealized P&L
    #   - Use for: Alpha research, strategy comparison
    #
    # COMPOUNDING_MODE = True (PRODUCTION MODE):
    #   - Positions sized on CURRENT portfolio value
    #   - Returns are geometrically compounded
    #   - Simulates real fund with reinvestment
    #   - Use for: Pitchbooks, AUM projections
    compounding_mode: bool = False
    
    # Leverage multiplier for risk scaling
    # 1.0 = 100% exposure when fully invested
    # 0.5 = 50% exposure (more conservative)
    # 2.0 = 200% exposure (requires margin)
    leverage: float = 1.0
    
    # Target allocation in low-volatility regime (Regime 0)
    # 1.0 = fully invested, 0.5 = half invested
    low_vol_allocation: float = 1.0
    
    # Target allocation in high-volatility regime (Regime 1)
    # 0.0 = fully cash, 0.5 = half invested
    high_vol_allocation: float = 0.0
    
    # -------------------------------------------------------------------------
    # TRANSACTION COSTS (IBKR-LIKE)
    # -------------------------------------------------------------------------
    # Realistic cost structure based on Interactive Brokers fee schedule
    # These costs significantly impact strategy performance and viability
    
    # Commission: per-share cost structure
    # IBKR Tiered: $0.0035 per share, min $0.35, max 1% of trade value
    commission_per_share: float = 0.0035
    commission_min: float = 0.35
    commission_max_pct: float = 0.01
    
    # Bid-ask spread as percentage (half-spread per side)
    # URTH is highly liquid; 1bp spread is conservative
    spread_bps: float = 1.0  # 0.01%
    
    # Market impact as percentage
    # Accounts for price movement when executing larger orders
    market_impact_bps: float = 2.0  # 0.02%
    
    # Margin interest rate for borrowed funds (short positions)
    # IBKR: Fed Funds + 1.5% for accounts > $100k
    margin_rate_annual: float = 0.06  # ~6% annually
    
    # -------------------------------------------------------------------------
    # WALK-FORWARD VALIDATION PARAMETERS
    # -------------------------------------------------------------------------
    # These parameters control the expanding window walk-forward process
    
    # Minimum training window before starting walk-forward
    # 504 days = 2 years of data for initial model estimation
    # Increased from V1.0 (252) to allow better optimization
    min_train_window: int = 504
    
    # Validation window size (for hyperparameter selection)
    # 126 days ~ 6 months for more stable OOS estimates
    # Increased from V1.0 (63) to reduce fold-level variance
    validation_window: int = 126
    
    # How often to refit the model during walk-forward
    # 126 days = semi-annual refitting
    refit_frequency: int = 126
    
    # Step size for walk-forward iteration
    # 63 days ~ quarterly steps
    step_size: int = 63
    
    # -------------------------------------------------------------------------
    # HYPERPARAMETER OPTIMIZATION PARAMETERS (NEW IN V2.0)
    # -------------------------------------------------------------------------
    # Grid search ranges for intra-fold optimization
    # These define the search space for parameter tuning
    
    # Volatility lookback grid (days)
    # Range: 10 (2 weeks) to 63 (3 months)
    # For fast testing, use [20, 40] instead
    vol_lookback_grid: List[int] = field(default_factory=lambda: [10, 15, 20, 30, 42, 63])
    
    # Quantile window grid (days)
    # Range: 126 (6 months) to 504 (2 years)
    # For fast testing, use [252] instead
    quantile_window_grid: List[int] = field(default_factory=lambda: [126, 189, 252, 378, 504])
    
    # Regime quantile grid
    # Range: 0.3 to 0.7 (avoid extremes)
    # For fast testing, use [0.4, 0.5, 0.6] instead
    regime_quantile_grid: List[float] = field(default_factory=lambda: [0.3, 0.4, 0.5, 0.6, 0.7])
    
    # Persistence days grid
    # Range: 1 to 5 days
    # For fast testing, use [2, 3] instead
    persistence_days_grid: List[int] = field(default_factory=lambda: [1, 2, 3, 4, 5])
    
    # FAST MODE: Reduced grid for quick testing (~10x faster)
    # Set to True to use minimal grids: [20,40], [252], [0.5], [3]
    fast_mode: bool = False
    
    # Optimization objective function
    # Options: 'sharpe', 'sortino', 'calmar', 'return_dd_ratio'
    optimization_objective: str = 'sharpe'
    
    # Maximum parameter combinations to evaluate (for efficiency)
    # If grid exceeds this, use random sampling
    max_param_combinations: int = 200
    
    # -------------------------------------------------------------------------
    # ROBUSTNESS TESTING PARAMETERS
    # -------------------------------------------------------------------------
    # Thresholds for statistical validation of strategy performance
    
    # Global Robustness Index threshold for deployment
    # > 0.65 indicates statistically robust strategy
    gri_threshold: float = 0.65
    
    # Minimum PSR for statistical significance
    # 0.95 = 95% confidence that true Sharpe > 0
    min_psr: float = 0.95
    
    # Walk-forward efficiency threshold
    # > 0.5 indicates OOS performance is at least 50% of IS
    min_wfe: float = 0.5
    
    # Number of trials for multiple testing adjustment (DSR calculation)
    # This should reflect the actual number of parameter combinations tested
    # V2.0: Dynamically calculated based on grid size
    num_trials: int = 100
    
    # Benchmark Sharpe for PSR calculation
    # 0 = test if strategy beats zero (standard)
    # Can be set to market Sharpe (~0.4) for more stringent test
    psr_benchmark_sharpe: float = 0.0
    
    # -------------------------------------------------------------------------
    # VISUALIZATION PARAMETERS
    # -------------------------------------------------------------------------
    
    # Enable/disable plot generation
    enable_plots: bool = True
    
    # Save plots to files
    save_plots: bool = True
    
    # Plot resolution
    dpi: int = 150
    
    # Figure dimensions
    figure_width: float = 16
    figure_height: float = 12
    
    # Color scheme for plots
    color_equity_curve: str = '#1f77b4'  # Blue
    color_benchmark: str = '#7f7f7f'     # Gray
    color_regime_low_vol: str = '#2ca02c'  # Green (invested)
    color_regime_high_vol: str = '#d62728'  # Red (cash)
    color_drawdown: str = '#ff7f0e'      # Orange
    color_in_sample: str = 'steelblue'
    color_out_of_sample: str = 'coral'


# ==============================================================================
# DATA ACQUISITION MODULE
# ==============================================================================
# Downloads daily Yahoo Finance prices
#
# UNCHANGED FROM V1.0 - Data acquisition is solid
# ==============================================================================

def fetch_historical_data(ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
    """Load daily OHLCV with prices adjusted for splits and dividends."""
    print(f"[DATA] Fetching {ticker} from {start_date} to {end_date}...")
    df = download_ohlcv(ticker, start_date, end_date, auto_adjust=True)

    actual_start = df.index.min()
    actual_end = df.index.max()
    requested_start = pd.to_datetime(start_date)
    requested_end = pd.to_datetime(end_date)

    if actual_start > requested_start + timedelta(days=5):
        print(f"[WARNING] Data for {ticker} starts late: {actual_start.date()}")
    if actual_end < requested_end - timedelta(days=5):
        print(f"[WARNING] Data for {ticker} ends early: {actual_end.date()}")

    print(f"[DATA] Retrieved {len(df)} trading days for {ticker}")
    return df


# ==============================================================================
# YANG-ZHANG VOLATILITY ESTIMATOR
# ==============================================================================
# Efficient volatility estimator using OHLC data
# Superior to close-to-close volatility in terms of estimation efficiency
#
# UNCHANGED FROM V1.0 - Volatility calculation is correct
# ==============================================================================

class YangZhangVolatility:
    """
    Yang-Zhang (2000) volatility estimator implementation.
    
    THEORETICAL BACKGROUND:
    ----------------------
    The Yang-Zhang estimator combines:
    1. Overnight volatility (open-to-previous close)
    2. Opening jump volatility (open variance)
    3. Close-to-close volatility (Rogers-Satchell component)
    
    ADVANTAGES:
    - Uses all OHLC information efficiently
    - Handles overnight gaps (important for ETFs)
    - Unbiased for continuous diffusion processes
    - More efficient than simple close-to-close
    
    IMPLEMENTATION NOTES:
    - Uses closed='left' rolling windows to prevent look-ahead bias
    - Annualization factor of 252 for trading days
    - Returns NaN for insufficient data (no forward/backward filling)
    """
    
    @staticmethod
    def calculate(
        df: pd.DataFrame, 
        window: int, 
        config: StrategyConfig
    ) -> pd.Series:
        """
        Calculate Yang-Zhang volatility using OHLC data.
        
        LAG OPTIMIZATION (V2.1):
        -----------------------
        Removed closed='left' from volatility calculation.
        - Volatility at T now includes data from T (not just T-1)
        - Combined with shift(1) in BacktestEngine = 1-day lag total
        - Result: Signal at Close[T] -> Execute at Open[T+1]
        
        NOTE: vol_threshold still uses closed='left' to prevent look-ahead
        in the quantile calculation.
        
        Args:
            df: DataFrame with Open, High, Low, Close columns
            window: Rolling window size
            config: Strategy configuration object
            
        Returns:
            Series of annualized Yang-Zhang volatility estimates
        """
        # Log price transformations
        log_open = np.log(df['Open'])
        log_high = np.log(df['High'])
        log_low = np.log(df['Low'])
        log_close = np.log(df['Close'])
        log_close_prev = log_close.shift(1)
        
        # Overnight variance - includes current day's data
        overnight_ret = log_open - log_close_prev
        var_overnight = overnight_ret.rolling(window).var()
        
        # Open-to-close variance - includes current day's data
        open_close_ret = log_close - log_open
        var_open_close = open_close_ret.rolling(window).var()
        
        # Rogers-Satchell variance - includes current day's data
        rs_component = (log_high - log_close) * (log_high - log_open) + \
                      (log_low - log_close) * (log_low - log_open)
        var_rs = rs_component.rolling(window).mean()
        
        # Optimal weighting factor k
        k = config.yz_k_numerator / (
            config.yz_k_denominator_constant + 
            (window + 1) / (window - 1)
        )
        
        # Combine components
        yz_variance = var_overnight + k * var_open_close + (1 - k) * var_rs
        yz_volatility = np.sqrt(yz_variance) * np.sqrt(config.yz_annualization_factor)
        
        return yz_volatility


# ==============================================================================
# THRESHOLD REGIME DETECTOR WITH PERSISTENCE FILTER
# ==============================================================================
# Implements causal (no look-ahead) regime detection using rolling quantiles
#
# KEY IMPROVEMENT V2.0:
# - Added signal persistence filter to reduce whipsaws
# - Configurable persistence_days parameter
# - Tracks regime duration for analysis
# ==============================================================================

class ThresholdRegimeDetector:
    """
    Threshold-based regime detection using rolling quantiles with persistence filter.
    
    V2.0 IMPROVEMENTS:
    -----------------
    - PERSISTENCE FILTER: Regime must persist N days before signal changes
    - This reduces whipsaw trades caused by temporary volatility spikes
    - Addresses V1.0 issue of 1015 trades in ~5 years
    
    METHODOLOGY:
    -----------
    1. Calculate volatility indicator (Yang-Zhang)
    2. Estimate regime threshold using ROLLING quantiles (not global)
    3. Assign raw regime based on current volatility vs threshold
    4. Apply persistence filter: only change signal after N days confirmation
    
    CAUSALITY GUARANTEE:
    -------------------
    - Rolling window uses closed='left' parameter
    - Quantile at time T uses only data from T-window to T-1
    - Persistence filter only looks at past regimes
    - No look-ahead bias in threshold estimation or signal generation
    """
    
    def __init__(
        self, 
        config: StrategyConfig,
        vol_lookback: Optional[int] = None,
        quantile_window: Optional[int] = None,
        regime_quantile: Optional[float] = None,
        persistence_days: Optional[int] = None
    ):
        """
        Initialize the regime detector with configuration.
        
        Args:
            config: Strategy configuration object
            vol_lookback: Override for volatility window (for optimization)
            quantile_window: Override for quantile window (for optimization)
            regime_quantile: Override for regime quantile (for optimization)
            persistence_days: Override for persistence filter (for optimization)
        """
        self.config = config
        
        # Use overrides if provided, otherwise use config defaults
        self.vol_lookback = vol_lookback or config.vol_lookback
        self.quantile_window = quantile_window or config.quantile_window
        self.regime_quantile = regime_quantile or config.regime_quantile
        self.persistence_days = persistence_days or config.persistence_days
        
    def detect_regimes(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Detect regimes using threshold model on volatility with persistence filter.
        
        PROCESSING STEPS:
        1. Calculate Yang-Zhang volatility
        2. Compute rolling quantile threshold
        3. Assign RAW regime labels based on volatility vs threshold
        4. Apply persistence filter to generate FILTERED signal
        5. Calculate additional regime statistics
        
        Args:
            df: DataFrame with OHLC price data
            
        Returns:
            DataFrame with original data plus:
            - 'volatility': Yang-Zhang volatility estimate
            - 'vol_threshold': Rolling quantile threshold
            - 'regime_raw': Raw regime label (before persistence)
            - 'regime': Filtered regime label (after persistence)
            - 'regime_name': Human-readable regime name
            - 'regime_duration': Days in current regime
        """
        df = df.copy()
        
        # ---------------------------------------------------------------------
        # STEP 1: Calculate Yang-Zhang volatility with specified lookback
        # ---------------------------------------------------------------------
        df['volatility'] = YangZhangVolatility.calculate(
            df, 
            self.vol_lookback, 
            self.config
        )
        
        # ---------------------------------------------------------------------
        # STEP 2: Calculate rolling quantile threshold
        # ---------------------------------------------------------------------
        # CRITICAL: closed='left' ONLY here (not in volatility calculation)
        # This compares today's volatility vs historical quantile distribution
        # - Volatility at T includes data up to T (no closed='left')
        # - Threshold at T uses quantile of volatility up to T-1 (closed='left')
        # - Combined with shift(1) in BacktestEngine = 1-day total lag
        # - Signal at Close[T] -> Execute at Open[T+1]
        df['vol_threshold'] = df['volatility'].rolling(
            self.quantile_window,
            closed='left'  # Excludes current observation from threshold calc
        ).quantile(self.regime_quantile)
        
        # ---------------------------------------------------------------------
        # STEP 3: Assign RAW regime labels (before persistence filter)
        # ---------------------------------------------------------------------
        # Regime 0: Low volatility (at or below threshold)
        # Regime 1: High volatility (above threshold)
        df['regime_raw'] = (df['volatility'] > df['vol_threshold']).astype(int)
        
        # ---------------------------------------------------------------------
        # STEP 4: Apply persistence filter (NEW IN V2.0)
        # ---------------------------------------------------------------------
        # Only change regime after it has persisted for N consecutive days
        # This reduces whipsaw trades from temporary regime changes
        
        if self.config.use_persistence_filter and self.persistence_days > 1:
            df['regime'] = self._apply_persistence_filter(
                df['regime_raw'], 
                self.persistence_days
            )
        else:
            df['regime'] = df['regime_raw']
        
        # ---------------------------------------------------------------------
        # STEP 5: Calculate regime statistics
        # ---------------------------------------------------------------------
        # Human-readable labels
        regime_names = {0: 'Low Vol', 1: 'High Vol'}
        df['regime_name'] = df['regime'].map(regime_names)
        
        # Calculate regime duration (days in current regime)
        df['regime_duration'] = self._calculate_regime_duration(df['regime'])
        
        # Daily returns using Adjusted Close
        df['returns'] = df['Close'].pct_change()
        df['log_returns'] = np.log(df['Close'] / df['Close'].shift(1))
        
        return df
    
    def _apply_persistence_filter(
        self, 
        raw_regime: pd.Series, 
        persistence_days: int
    ) -> pd.Series:
        """
        Apply persistence filter to raw regime signals.
        
        LOGIC:
        - Start with first valid regime
        - Only switch regime after N consecutive days of the new regime
        - This filters out short-term noise
        
        EXAMPLE (persistence_days=3):
        Raw:      0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0
        Filtered: 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0
        
        The regime only changes after 3 consecutive days of the new regime.
        
        Args:
            raw_regime: Series of raw regime labels
            persistence_days: Minimum days regime must persist
            
        Returns:
            Series of filtered regime labels
        """
        filtered = raw_regime.copy()
        
        # Get valid (non-NaN) portion
        valid_mask = ~raw_regime.isna()
        valid_indices = raw_regime[valid_mask].index
        
        if len(valid_indices) < persistence_days:
            return filtered
        
        # Initialize with first valid regime
        current_regime = raw_regime.loc[valid_indices[0]]
        consecutive_count = 1
        
        for i, idx in enumerate(valid_indices):
            raw_value = raw_regime.loc[idx]
            
            if pd.isna(raw_value):
                filtered.loc[idx] = np.nan
                continue
            
            if raw_value == current_regime:
                # Same regime, keep counting
                consecutive_count += 1
                filtered.loc[idx] = current_regime
            else:
                # Different regime, check if threshold met
                # Count consecutive days of new regime
                new_regime_count = 1
                for j in range(i + 1, min(i + persistence_days, len(valid_indices))):
                    future_idx = valid_indices[j]
                    if raw_regime.loc[future_idx] == raw_value:
                        new_regime_count += 1
                    else:
                        break
                
                # Only switch if we have enough consecutive days
                # (looking at past data only - check if previous N-1 days were same)
                past_count = 0
                for j in range(max(0, i - persistence_days + 1), i):
                    past_idx = valid_indices[j]
                    if raw_regime.loc[past_idx] == raw_value:
                        past_count += 1
                
                if past_count >= persistence_days - 1:
                    current_regime = raw_value
                    consecutive_count = 1
                    filtered.loc[idx] = current_regime
                else:
                    filtered.loc[idx] = current_regime
        
        return filtered
    
    def _calculate_regime_duration(self, regime: pd.Series) -> pd.Series:
        """
        Calculate how many days the current regime has persisted.
        
        Args:
            regime: Series of regime labels
            
        Returns:
            Series of regime duration (days)
        """
        duration = pd.Series(index=regime.index, dtype=float)
        
        current_count = 0
        prev_regime = None
        
        for idx, val in regime.items():
            if pd.isna(val):
                duration.loc[idx] = np.nan
                continue
            
            if val == prev_regime:
                current_count += 1
            else:
                current_count = 1
                prev_regime = val
            
            duration.loc[idx] = current_count
        
        return duration
    
    def get_signal(self, regime: int) -> float:
        """
        Convert regime to target allocation signal.
        
        Args:
            regime: Integer regime label (0 or 1)
            
        Returns:
            Target allocation as fraction of capital
        """
        if pd.isna(regime):
            return np.nan
        
        if regime == 0:  # Low volatility
            return self.config.low_vol_allocation * self.config.leverage
        else:  # High volatility
            return self.config.high_vol_allocation * self.config.leverage


# ==============================================================================
# TRANSACTION COST CALCULATOR
# ==============================================================================
# Implements realistic IBKR-like cost structure
#
# UNCHANGED FROM V1.0 - Cost calculation is correct
# ==============================================================================

class TransactionCostCalculator:
    """
    Calculates realistic transaction costs for backtesting.
    
    COST COMPONENTS:
    ---------------
    1. Commission: Per-share cost with min/max bounds
    2. Bid-Ask Spread: Cost of crossing the spread
    3. Market Impact: Price movement from trade execution
    4. Margin Interest: Cost of borrowed funds
    """
    
    def __init__(self, config: StrategyConfig):
        self.config = config
        
    def calculate_commission(self, shares: float, price: float) -> float:
        """Calculate IBKR-style commission."""
        shares = abs(shares)
        trade_value = shares * price
        
        raw_commission = shares * self.config.commission_per_share
        min_commission = self.config.commission_min
        max_commission = trade_value * self.config.commission_max_pct
        
        return max(min_commission, min(raw_commission, max_commission))
    
    def calculate_spread_cost(self, shares: float, price: float) -> float:
        """Calculate bid-ask spread cost."""
        trade_value = abs(shares) * price
        return trade_value * (self.config.spread_bps / 10000)
    
    def calculate_market_impact(self, shares: float, price: float) -> float:
        """Calculate market impact cost."""
        trade_value = abs(shares) * price
        return trade_value * (self.config.market_impact_bps / 10000)
    
    def calculate_total_cost(self, shares: float, price: float) -> float:
        """Calculate total transaction cost."""
        commission = self.calculate_commission(shares, price)
        spread = self.calculate_spread_cost(shares, price)
        impact = self.calculate_market_impact(shares, price)
        return commission + spread + impact
    
    def calculate_margin_interest(self, borrowed_value: float, days: int = 1) -> float:
        """Calculate margin interest on borrowed funds."""
        daily_rate = self.config.margin_rate_annual / 252
        return abs(borrowed_value) * daily_rate * days


# ==============================================================================
# BACKTESTING ENGINE
# ==============================================================================
# Implements hybrid P&L calculation with proper temporal alignment
#
# IMPROVEMENTS FROM V1.0:
# - Better handling of signal transitions
# - More accurate P&L calculation
# - Additional diagnostics for fold analysis
# ==============================================================================

class BacktestEngine:
    """
    Production-grade backtesting engine with hybrid P&L calculation.
    
    V2.1 CRITICAL FIX:
    -----------------
    PROBLEM (V2.0): Daily rebalancing to maintain fixed $100k notional caused
    spurious trades every day (491 trades in 504 days). This was NOT regime
    trading - it was daily price-drift rebalancing.
    
    SOLUTION: Trade ONLY on signal changes (regime transitions).
    Let position "drift" with market when regime is unchanged.
    
    CORRECT BEHAVIOR:
    - Signal changes (regime transition) -> Execute full rebalance to target
    - Same signal (holding) -> Do nothing, let position float
    
    This reduces trades from ~500/year to ~10-30/year (actual regime changes).
    
    HYBRID P&L METHODOLOGY:
    ----------------------
    - Execution Days (signal change): Position sized at Open[T], P&L from overnight gap
    - Holding Days (no change): Position drifts, P&L = shares * price change
    
    TEMPORAL INTEGRITY:
    ------------------
    Signal at T-1 Close -> Execute at Open[T] ONLY if signal changed
    """
    
    def __init__(self, config: StrategyConfig):
        self.config = config
        self.cost_calculator = TransactionCostCalculator(config)
        
    def run_backtest(
        self, 
        df: pd.DataFrame, 
        signals: pd.Series,
        return_diagnostics: bool = False
    ) -> pd.DataFrame:
        """
        Execute the backtest with signal-change-only execution.
        
        CRITICAL FIX (V2.1):
        -------------------
        Only execute trades when the signal (target allocation) CHANGES.
        Do NOT rebalance daily to maintain fixed notional - this was causing
        spurious trades on every price movement.
        
        Args:
            df: DataFrame with OHLC data
            signals: Series of target allocations (0 to 1)
            return_diagnostics: If True, include additional diagnostic columns
            
        Returns:
            DataFrame with complete backtest results
        """
        df = df.copy()
        
        # Lag signals: signal generated at T is executed at T+1
        df['target_allocation'] = signals.shift(1)
        
        # Initialize storage
        results = {
            'date': [],
            'open_price': [],
            'close_price': [],
            'target_alloc': [],
            'actual_alloc': [],
            'shares': [],
            'position_value': [],
            'cash': [],
            'portfolio_value': [],
            'gross_return': [],
            'transaction_cost': [],
            'net_return': [],
            'benchmark_return': [],
            'is_execution_day': [],
            'regime': [],
            'signal_change': []
        }
        
        # Initialize portfolio state
        current_cash = self.config.initial_capital
        current_shares = 0.0
        cash_reserve = 0.0  # Tracks cumulative P&L in research mode
        
        # CRITICAL: Track the SIGNAL that was last executed, not the shares
        # This determines when we actually need to trade
        executed_signal = None  # None = no position yet
        
        # Get valid trading dates
        valid_dates = df.dropna(subset=['target_allocation', 'Open', 'Close']).index
        
        if len(valid_dates) == 0:
            raise ValueError("No valid trading dates after warm-up period")
        
        previous_close = df.loc[valid_dates[0], 'Close']
        
        # Main backtest loop
        for i, date in enumerate(valid_dates):
            open_price = df.loc[date, 'Open']
            close_price = df.loc[date, 'Close']
            target_alloc = df.loc[date, 'target_allocation']
            regime = df.loc[date, 'regime'] if 'regime' in df.columns else np.nan
            
            # -----------------------------------------------------------------
            # CRITICAL FIX: Detect SIGNAL CHANGE, not position drift
            # -----------------------------------------------------------------
            # We only trade when the target allocation changes significantly
            # NOT when the position value drifts due to price movements
            
            if executed_signal is None:
                # First day: need to establish position
                signal_changed = True
            else:
                # Check if signal (allocation target) has changed
                signal_changed = abs(target_alloc - executed_signal) > 0.01
            
            # -----------------------------------------------------------------
            # EXECUTION LOGIC: Only trade on signal change
            # -----------------------------------------------------------------
            is_execution_day = signal_changed
            transaction_cost = 0.0
            
            if signal_changed:
                # CASE 1: SIGNAL CHANGED -> Execute full rebalance to new target
                # Calculate target position based on initial capital (research mode)
                # or current portfolio value (production mode)
                
                if self.config.compounding_mode:
                    # Production mode: size on current portfolio value
                    current_portfolio_value = self.config.initial_capital + cash_reserve
                    sizing_capital = current_portfolio_value
                else:
                    # Research mode: always size on initial capital
                    sizing_capital = self.config.initial_capital
                
                target_value = target_alloc * sizing_capital * self.config.leverage
                new_shares = target_value / open_price if open_price > 0 else 0
                
                # Calculate shares to trade
                shares_change = new_shares - current_shares
                
                # Calculate transaction costs on the trade
                if abs(shares_change) > 0.001:
                    transaction_cost = self.cost_calculator.calculate_total_cost(
                        shares_change, 
                        open_price
                    )
                
                # Calculate P&L for the day:
                # Old position earns overnight return (close[t-1] to open[t])
                # Then we rebalance at open[t]
                # New position earns intraday return (open[t] to close[t])
                
                if i > 0 and current_shares > 0:
                    # Old position overnight return
                    overnight_return = (open_price - previous_close) * current_shares
                else:
                    overnight_return = 0.0
                
                # Intraday return on new position
                intraday_return = (close_price - open_price) * new_shares
                
                position_return = overnight_return + intraday_return
                
                # Update shares and executed signal
                current_shares = new_shares
                executed_signal = target_alloc
                
            else:
                # CASE 2: NO SIGNAL CHANGE -> Do nothing, let position drift
                # This is the critical fix: we don't rebalance just because
                # the position value changed due to price movements
                
                if current_shares > 0:
                    # Holding: P&L = shares * (close - previous_close)
                    position_return = current_shares * (close_price - previous_close)
                else:
                    # Flat (in cash): no P&L
                    position_return = 0.0
            
            # -----------------------------------------------------------------
            # Update portfolio value
            # -----------------------------------------------------------------
            position_value = current_shares * close_price
            
            gross_return = position_return
            net_return = gross_return - transaction_cost
            cash_reserve += net_return
            
            current_portfolio_value = self.config.initial_capital + cash_reserve
            current_cash = current_portfolio_value - position_value
            
            # Benchmark return (buy and hold)
            if previous_close > 0:
                benchmark_return = close_price / previous_close - 1
            else:
                benchmark_return = 0.0
            
            # -----------------------------------------------------------------
            # Record results
            # -----------------------------------------------------------------
            results['date'].append(date)
            results['open_price'].append(open_price)
            results['close_price'].append(close_price)
            results['target_alloc'].append(target_alloc)
            results['actual_alloc'].append(position_value / current_portfolio_value if current_portfolio_value > 0 else 0)
            results['shares'].append(current_shares)
            results['position_value'].append(position_value)
            results['cash'].append(current_cash)
            results['portfolio_value'].append(current_portfolio_value)
            results['gross_return'].append(gross_return)
            results['transaction_cost'].append(transaction_cost)
            results['net_return'].append(net_return)
            results['benchmark_return'].append(benchmark_return)
            results['is_execution_day'].append(is_execution_day)
            results['regime'].append(regime)
            results['signal_change'].append(signal_changed)
            
            # Update state for next iteration
            previous_close = close_price
        
        # Create results DataFrame
        results_df = pd.DataFrame(results)
        results_df.set_index('date', inplace=True)
        
        # Calculate cumulative metrics
        results_df['cumulative_return'] = (
            results_df['portfolio_value'] / self.config.initial_capital - 1
        )
        results_df['cumulative_benchmark'] = (
            (1 + results_df['benchmark_return']).cumprod() - 1
        )
        results_df['strategy_return'] = results_df['portfolio_value'].pct_change()
        
        return results_df


# ==============================================================================
# PERFORMANCE ANALYTICS
# ==============================================================================
# Calculates comprehensive performance metrics including robustness measures
#
# V2.0 IMPROVEMENTS:
# - Fixed DSR calculation (was returning 0 incorrectly)
# - Better handling of edge cases
# - More accurate PSR calculation
# ==============================================================================

class PerformanceAnalytics:
    """
    Comprehensive performance analysis and robustness testing.
    
    V2.0 IMPROVEMENTS:
    -----------------
    - FIXED DSR CALCULATION: V1.0 returned 0 due to incorrect formula
    - Better numerical stability for extreme cases
    - Improved handling of short return series
    """
    
    def __init__(self, config: StrategyConfig):
        self.config = config
        
    def calculate_basic_metrics(
        self, 
        returns: pd.Series, 
        benchmark_returns: Optional[pd.Series] = None,
        annualization_factor: int = 252
    ) -> Dict[str, float]:
        """
        Calculate basic performance metrics.
        
        Args:
            returns: Series of daily returns
            benchmark_returns: Optional benchmark returns
            annualization_factor: Trading days per year
            
        Returns:
            Dictionary of performance metrics
        """
        returns = returns.dropna()
        
        if len(returns) < 20:
            return {
                'error': 'Insufficient data',
                'total_return': 0,
                'annual_return': 0,
                'volatility': 0,
                'sharpe_ratio': 0,
                'sortino_ratio': 0,
                'max_drawdown': 0,
                'calmar_ratio': 0,
                'num_observations': len(returns)
            }
        
        # Basic statistics
        total_return = (1 + returns).prod() - 1
        n_years = len(returns) / annualization_factor
        annual_return = (1 + total_return) ** (1 / n_years) - 1 if n_years > 0 else 0
        volatility = returns.std() * np.sqrt(annualization_factor)
        
        # Sharpe Ratio
        sharpe = annual_return / volatility if volatility > 0 else 0
        
        # Sortino Ratio
        downside_returns = returns[returns < 0]
        downside_deviation = downside_returns.std() * np.sqrt(annualization_factor) if len(downside_returns) > 0 else 0
        sortino = annual_return / downside_deviation if downside_deviation > 0 else 0
        
        # Drawdown analysis
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.cummax()
        drawdown = (cumulative - running_max) / running_max
        max_drawdown = drawdown.min()
        
        # Calmar Ratio
        calmar = annual_return / abs(max_drawdown) if max_drawdown != 0 else 0
        
        # VaR and CVaR
        var_95 = returns.quantile(0.05)
        cvar_95 = returns[returns <= var_95].mean() if len(returns[returns <= var_95]) > 0 else var_95
        
        # Win rate
        win_rate = (returns > 0).sum() / len(returns)
        
        # Skewness and Kurtosis
        skewness = returns.skew()
        kurtosis = returns.kurtosis() + 3  # Convert Fisher's to standard
        
        metrics = {
            'total_return': total_return,
            'annual_return': annual_return,
            'volatility': volatility,
            'sharpe_ratio': sharpe,
            'sortino_ratio': sortino,
            'max_drawdown': max_drawdown,
            'calmar_ratio': calmar,
            'var_95': var_95,
            'cvar_95': cvar_95,
            'win_rate': win_rate,
            'skewness': skewness,
            'kurtosis': kurtosis,
            'num_observations': len(returns)
        }
        
        # Benchmark comparison
        if benchmark_returns is not None:
            benchmark_returns = benchmark_returns.reindex(returns.index).dropna()
            if len(benchmark_returns) > 20:
                benchmark_total = (1 + benchmark_returns).prod() - 1
                excess_return = total_return - benchmark_total
                
                # Tracking error and Information Ratio
                active_returns = returns - benchmark_returns.reindex(returns.index).fillna(0)
                tracking_error = active_returns.std() * np.sqrt(annualization_factor)
                
                benchmark_n_years = len(benchmark_returns) / annualization_factor
                benchmark_annual = (1 + benchmark_total) ** (1 / benchmark_n_years) - 1 if benchmark_n_years > 0 else 0
                info_ratio = (annual_return - benchmark_annual) / tracking_error if tracking_error > 0 else 0
                
                metrics['benchmark_return'] = benchmark_total
                metrics['excess_return'] = excess_return
                metrics['tracking_error'] = tracking_error
                metrics['information_ratio'] = info_ratio
        
        return metrics
    
    def calculate_deflated_sharpe_ratio(
        self, 
        sharpe: float, 
        num_observations: int, 
        num_trials: int,
        skewness: float = 0,
        kurtosis: float = 3
    ) -> float:
        """
        Calculate Deflated Sharpe Ratio (DSR).
        
        V2.0 FIX:
        --------
        The V1.0 implementation incorrectly calculated the expected max Sharpe
        and the variance term, leading to DSR=0 in many cases.
        
        This implementation follows Bailey & Lopez de Prado (2014) more closely.
        
        DSR = P(SR* > SR_max) where SR_max is the expected max Sharpe under H0
        
        Args:
            sharpe: Observed Sharpe ratio (annualized)
            num_observations: Number of return observations
            num_trials: Number of independent trials/tests
            skewness: Skewness of returns
            kurtosis: Kurtosis of returns (standard, not excess)
            
        Returns:
            DSR probability (0 to 1)
        """
        if num_trials <= 1:
            return self.calculate_probabilistic_sharpe_ratio(
                sharpe, num_observations, 0, skewness, kurtosis
            )
        
        # Expected maximum Sharpe ratio under null hypothesis (all strategies have SR=0)
        # Using the Euler-Mascheroni approximation for expected max of N standard normals
        # E[max(Z_1, ..., Z_N)] ≈ sqrt(2 * log(N)) - (gamma + log(pi)) / (2 * sqrt(2 * log(N)))
        # Simplified: E[max] ≈ sqrt(2 * log(N)) for large N
        
        try:
            if num_trials > 1:
                # More accurate formula from Bailey & Lopez de Prado
                euler_mascheroni = 0.5772156649
                z_arg = 1 - 1 / num_trials
                z_arg = max(0.0001, min(0.9999, z_arg))  # Bound for numerical stability
                z_quantile = stats.norm.ppf(z_arg)
                
                e_max_arg = 1 - 1 / (num_trials * np.e)
                e_max_arg = max(0.0001, min(0.9999, e_max_arg))
                
                expected_max_sr = (1 - euler_mascheroni) * stats.norm.ppf(z_arg) + \
                                  euler_mascheroni * stats.norm.ppf(e_max_arg)
                
                # Scale by standard error of Sharpe estimator
                # This converts the expected max of standardized SRs to actual SR scale
                sr_std = np.sqrt((1 + 0.5 * sharpe**2 - skewness * sharpe + 
                                 (kurtosis - 3) / 4 * sharpe**2) / num_observations)
                
                expected_max_sr = expected_max_sr * sr_std
            else:
                expected_max_sr = 0
                sr_std = np.sqrt((1 + 0.5 * sharpe**2) / num_observations)
            
            # DSR: probability that observed SR exceeds expected max under H0
            if sr_std > 0:
                # Z-score of observed SR vs expected max
                z_score = (sharpe - expected_max_sr) / sr_std
                dsr = stats.norm.cdf(z_score)
            else:
                dsr = 0.5
            
            # Bound result
            dsr = max(0, min(1, dsr))
            
        except Exception as e:
            print(f"[WARNING] DSR calculation error: {e}")
            dsr = 0.5
        
        return dsr
    
    def calculate_probabilistic_sharpe_ratio(
        self, 
        sharpe: float, 
        num_observations: int,
        benchmark_sharpe: float = 0,
        skewness: float = 0,
        kurtosis: float = 3
    ) -> float:
        """
        Calculate Probabilistic Sharpe Ratio (PSR).
        
        PSR = P(SR_true > benchmark_sharpe)
        
        Args:
            sharpe: Observed Sharpe ratio
            num_observations: Number of observations
            benchmark_sharpe: Benchmark Sharpe to compare against
            skewness: Skewness of returns
            kurtosis: Kurtosis of returns (standard, not excess)
            
        Returns:
            Probability that true Sharpe > benchmark Sharpe
        """
        if num_observations < 2:
            return 0.5
        
        # Standard error of Sharpe ratio estimator (Lo, 2002)
        # SE(SR) = sqrt((1 + 0.5*SR^2 - γ3*SR + (γ4-3)/4*SR^2) / N)
        # where γ3 = skewness, γ4 = kurtosis
        
        variance_term = (1 + 0.5 * sharpe**2 - skewness * sharpe + 
                        (kurtosis - 3) / 4 * sharpe**2)
        
        if variance_term <= 0:
            variance_term = 1  # Fallback for numerical issues
        
        sharpe_std_error = np.sqrt(variance_term / num_observations)
        
        if sharpe_std_error > 0:
            z_score = (sharpe - benchmark_sharpe) / sharpe_std_error
            psr = stats.norm.cdf(z_score)
        else:
            psr = 0.5
        
        return max(0, min(1, psr))
    
    def calculate_hurst_exponent(
        self, 
        returns: pd.Series, 
        max_lag: int = 100
    ) -> float:
        """
        Calculate Hurst exponent using R/S analysis.
        
        H = 0.5: Random walk
        H > 0.5: Trending/persistent
        H < 0.5: Mean-reverting
        """
        returns = returns.dropna().values
        n = len(returns)
        
        if n < 50:
            return 0.5
        
        max_lag = min(max_lag, n // 4)
        lags = range(10, max_lag)
        rs_values = []
        
        for lag in lags:
            num_chunks = n // lag
            rs_list = []
            
            for i in range(num_chunks):
                chunk = returns[i*lag:(i+1)*lag]
                mean_chunk = chunk.mean()
                cum_dev = np.cumsum(chunk - mean_chunk)
                r = cum_dev.max() - cum_dev.min()
                s = chunk.std()
                
                if s > 0:
                    rs_list.append(r / s)
            
            if len(rs_list) > 0:
                rs_values.append((np.log(lag), np.log(np.mean(rs_list))))
        
        if len(rs_values) < 2:
            return 0.5
        
        rs_array = np.array(rs_values)
        slope, _, _, _, _ = stats.linregress(rs_array[:, 0], rs_array[:, 1])
        
        return max(0, min(1, slope))
    
    def calculate_cdar(
        self, 
        returns: pd.Series, 
        alpha: float = 0.05
    ) -> float:
        """
        Calculate Conditional Drawdown at Risk (CDaR).
        
        CDaR = E[DD | DD <= DD_alpha]
        """
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.cummax()
        drawdowns = (cumulative - running_max) / running_max
        
        threshold = drawdowns.quantile(alpha)
        worst_drawdowns = drawdowns[drawdowns <= threshold]
        
        if len(worst_drawdowns) > 0:
            return worst_drawdowns.mean()
        else:
            return drawdowns.min()
    
    def calculate_rolling_ic(
        self, 
        signals: pd.Series, 
        forward_returns: pd.Series, 
        window: int = 63
    ) -> pd.Series:
        """
        Calculate rolling Information Coefficient (IC).
        
        IC = correlation(signal, forward_return)
        """
        aligned_signals = signals.shift(1)
        rolling_ic = aligned_signals.rolling(window).corr(forward_returns)
        return rolling_ic
    
    def calculate_parameter_stability_index(
        self, 
        sharpes_by_fold: List[float]
    ) -> float:
        """
        Calculate Parameter Stability Index (PSI).
        
        V2.0: Uses fold-level Sharpes instead of parameter variations
        """
        if len(sharpes_by_fold) < 2:
            return 1.0
        
        sharpes = np.array([s for s in sharpes_by_fold if not np.isnan(s) and s != 0])
        
        if len(sharpes) < 2:
            return 0.5
        
        mean_sharpe = np.mean(sharpes)
        std_sharpe = np.std(sharpes)
        
        if mean_sharpe != 0:
            cv = abs(std_sharpe / mean_sharpe)
            psi = 1 / (1 + cv)
        else:
            psi = 0.5 if std_sharpe == 0 else 0.3
        
        return psi
    
    def calculate_global_robustness_index(
        self, 
        metrics: Dict[str, float]
    ) -> float:
        """
        Calculate Global Robustness Index (GRI).
        
        GRI combines multiple robustness metrics into a single score.
        """
        weights = {
            'dsr': 0.20,
            'psr': 0.20,
            'wfe': 0.20,
            'hurst': 0.10,
            'cdar': 0.15,
            'psi': 0.15
        }
        
        normalized = {}
        
        # DSR: already in [0, 1]
        normalized['dsr'] = max(0, min(1, metrics.get('dsr', 0.5)))
        
        # PSR: already in [0, 1]
        normalized['psr'] = max(0, min(1, metrics.get('psr', 0.5)))
        
        # WFE: cap at 1.5, floor at 0, map to [0, 1]
        wfe = metrics.get('wfe', 0.5)
        normalized['wfe'] = max(0, min(1, wfe / 1.5))
        
        # Hurst: already in [0, 1]
        normalized['hurst'] = max(0, min(1, metrics.get('hurst', 0.5)))
        
        # CDaR: normalize to [0, 1] where smaller (less negative) is better
        cdar = metrics.get('cdar', -0.2)
        normalized['cdar'] = max(0, min(1, 1 + cdar * 2))
        
        # PSI: already in [0, 1]
        normalized['psi'] = max(0, min(1, metrics.get('psi', 0.5)))
        
        gri = sum(weights[k] * normalized[k] for k in weights)
        
        return gri


# ==============================================================================
# HYPERPARAMETER OPTIMIZER
# ==============================================================================
# Performs intra-fold parameter optimization for walk-forward validation
#
# NEW IN V2.0:
# - Grid search over parameter space
# - Objective function selection (Sharpe, Sortino, Calmar)
# - Tracks optimal parameters per fold for stability analysis
# ==============================================================================

class HyperparameterOptimizer:
    """
    Intra-fold hyperparameter optimization for walk-forward validation.
    
    V2.0 FEATURE:
    ------------
    V1.0 used fixed parameters across all folds, which only tested one
    configuration. V2.0 optimizes parameters within each training window,
    testing the CONCEPT's robustness rather than a single parameter set.
    
    METHODOLOGY:
    -----------
    1. Generate parameter grid from configuration
    2. For each combination, run backtest on training data
    3. Select parameters that maximize objective function
    4. Apply optimal parameters to validation window
    
    OPTIMIZATION OBJECTIVES:
    - 'sharpe': Maximize Sharpe Ratio (default)
    - 'sortino': Maximize Sortino Ratio
    - 'calmar': Maximize Calmar Ratio
    - 'return_dd_ratio': Maximize Return / Max DD
    """
    
    def __init__(self, config: StrategyConfig):
        self.config = config
        self.analytics = PerformanceAnalytics(config)
        
    def generate_param_grid(self) -> List[Dict[str, Any]]:
        """
        Generate parameter combinations for grid search.
        
        If fast_mode is enabled, uses minimal grids for quick testing.
        
        Returns:
            List of parameter dictionaries
        """
        # Use reduced grids in fast mode
        if self.config.fast_mode:
            vol_lookbacks = [20, 40]
            quantile_windows = [252]
            regime_quantiles = [0.5]
            persistence_days_list = [3]
            print("    [FAST MODE] Using reduced parameter grid")
        else:
            vol_lookbacks = self.config.vol_lookback_grid
            quantile_windows = self.config.quantile_window_grid
            regime_quantiles = self.config.regime_quantile_grid
            persistence_days_list = self.config.persistence_days_grid
        
        # Generate all combinations
        all_params = []
        for vl, qw, rq, pd in product(vol_lookbacks, quantile_windows, 
                                      regime_quantiles, persistence_days_list):
            # Validate: quantile_window must be >= vol_lookback
            if qw >= vl:
                all_params.append({
                    'vol_lookback': vl,
                    'quantile_window': qw,
                    'regime_quantile': rq,
                    'persistence_days': pd
                })
        
        # If too many combinations, sample randomly
        if len(all_params) > self.config.max_param_combinations:
            np.random.seed(42)
            indices = np.random.choice(
                len(all_params), 
                self.config.max_param_combinations, 
                replace=False
            )
            all_params = [all_params[i] for i in indices]
        
        return all_params
    
    def evaluate_params(
        self, 
        params: Dict[str, Any], 
        df: pd.DataFrame,
        config: StrategyConfig
    ) -> Dict[str, float]:
        """
        Evaluate a single parameter combination on training data.
        
        Args:
            params: Parameter dictionary
            df: Training data
            config: Strategy configuration
            
        Returns:
            Dictionary with performance metrics
        """
        try:
            # Create regime detector with specified parameters
            detector = ThresholdRegimeDetector(
                config,
                vol_lookback=params['vol_lookback'],
                quantile_window=params['quantile_window'],
                regime_quantile=params['regime_quantile'],
                persistence_days=params['persistence_days']
            )
            
            # Detect regimes
            df_with_regimes = detector.detect_regimes(df)
            
            # Check for valid regimes
            valid_regimes = df_with_regimes['regime'].dropna()
            if len(valid_regimes) < 60:  # Need at least ~3 months
                return {'sharpe_ratio': -999, 'valid': False}
            
            # Generate signals
            signals = df_with_regimes['regime'].apply(detector.get_signal)
            
            # Run backtest
            backtest_engine = BacktestEngine(config)
            results = backtest_engine.run_backtest(df_with_regimes, signals)
            
            # Calculate metrics
            returns = results['strategy_return'].dropna()
            if len(returns) < 30:
                return {'sharpe_ratio': -999, 'valid': False}
            
            metrics = self.analytics.calculate_basic_metrics(returns)
            metrics['valid'] = True
            metrics['num_trades'] = results['is_execution_day'].sum()
            
            return metrics
            
        except Exception as e:
            return {'sharpe_ratio': -999, 'valid': False, 'error': str(e)}
    
    def optimize(
        self, 
        df: pd.DataFrame,
        verbose: bool = False
    ) -> Tuple[Dict[str, Any], Dict[str, float]]:
        """
        Find optimal parameters using grid search.
        
        Args:
            df: Training data
            verbose: Print progress
            
        Returns:
            Tuple of (optimal_params, optimal_metrics)
        """
        param_grid = self.generate_param_grid()
        
        if verbose:
            print(f"    Testing {len(param_grid)} parameter combinations...")
        
        best_params = None
        best_metrics = {'sharpe_ratio': -999}
        best_objective = -np.inf
        
        for i, params in enumerate(param_grid):
            metrics = self.evaluate_params(params, df, self.config)
            
            if not metrics.get('valid', False):
                continue
            
            # Get objective value
            if self.config.optimization_objective == 'sharpe':
                objective = metrics.get('sharpe_ratio', -999)
            elif self.config.optimization_objective == 'sortino':
                objective = metrics.get('sortino_ratio', -999)
            elif self.config.optimization_objective == 'calmar':
                objective = metrics.get('calmar_ratio', -999)
            elif self.config.optimization_objective == 'return_dd_ratio':
                ret = metrics.get('total_return', 0)
                dd = abs(metrics.get('max_drawdown', -1))
                objective = ret / dd if dd > 0 else -999
            else:
                objective = metrics.get('sharpe_ratio', -999)
            
            if objective > best_objective:
                best_objective = objective
                best_params = params
                best_metrics = metrics
        
        if best_params is None:
            # Fallback to default parameters
            best_params = {
                'vol_lookback': self.config.vol_lookback,
                'quantile_window': self.config.quantile_window,
                'regime_quantile': self.config.regime_quantile,
                'persistence_days': self.config.persistence_days
            }
        
        if verbose:
            print(f"    Best params: vol_lb={best_params['vol_lookback']}, "
                  f"q_win={best_params['quantile_window']}, "
                  f"q={best_params['regime_quantile']}, "
                  f"pers={best_params['persistence_days']}")
            print(f"    Best objective ({self.config.optimization_objective}): "
                  f"{best_objective:.3f}")
        
        return best_params, best_metrics


# ==============================================================================
# WALK-FORWARD VALIDATION ENGINE
# ==============================================================================
# Implements expanding window walk-forward with intra-fold optimization
#
# V2.0 IMPROVEMENTS:
# - Intra-fold parameter optimization
# - Better fold diagnostics
# - Parameter stability tracking
# - Zero-Sharpe fold analysis
# ==============================================================================

class WalkForwardValidator:
    """
    Walk-Forward Validation with expanding windows and intra-fold optimization.
    
    V2.0 IMPROVEMENTS:
    -----------------
    - INTRA-FOLD OPTIMIZATION: Parameters optimized within each training window
    - PARAMETER STABILITY: Tracks optimal parameters across folds
    - ZERO-SHARPE DIAGNOSTICS: Identifies why some folds have no activity
    - IMPROVED METRICS: Fixed DSR calculation
    
    METHODOLOGY:
    -----------
    For each fold:
    1. TRAIN: Optimize parameters on training data
    2. VALIDATE: Apply optimal parameters to validation data
    3. RECORD: Store both IS and OOS metrics for analysis
    
    The training window expands each fold while validation
    window stays fixed size.
    """
    
    def __init__(self, config: StrategyConfig):
        self.config = config
        self.analytics = PerformanceAnalytics(config)
        self.optimizer = HyperparameterOptimizer(config)
        
    def run_walk_forward(
        self, 
        df: pd.DataFrame,
        verbose: bool = True
    ) -> Dict[str, Any]:
        """
        Execute walk-forward validation with intra-fold optimization.
        
        Args:
            df: DataFrame with OHLC data
            verbose: Print progress
            
        Returns:
            Dictionary containing:
            - fold_results: List of per-fold metrics
            - aggregate_metrics: Overall robustness metrics
            - oos_returns: Combined out-of-sample returns
            - param_history: Parameters selected in each fold
        """
        # ---------------------------------------------------------------------
        # STEP 1: Initial regime detection for warm-up calculation
        # ---------------------------------------------------------------------
        # Use default parameters to find valid start point
        default_detector = ThresholdRegimeDetector(self.config)
        df_initial = default_detector.detect_regimes(df)
        
        # Find first valid index (after warm-up for largest possible window)
        max_warmup = max(self.config.quantile_window_grid) + max(self.config.vol_lookback_grid)
        
        valid_indices = df_initial.dropna(subset=['regime', 'vol_threshold']).index
        
        min_required = self.config.min_train_window + self.config.validation_window + max_warmup
        
        if len(valid_indices) < min_required:
            raise ValueError(f"Insufficient data after warm-up. "
                           f"Need {min_required}, have {len(valid_indices)}")
        
        # Initialize storage
        fold_results = []
        oos_returns_list = []
        param_history = []
        
        # Calculate number of trials for DSR (based on grid size)
        num_trials = len(self.optimizer.generate_param_grid())
        
        # ---------------------------------------------------------------------
        # STEP 2: Walk-forward loop
        # ---------------------------------------------------------------------
        fold_num = 0
        train_end_idx = self.config.min_train_window
        
        if verbose:
            print("\n" + "="*80)
            print("WALK-FORWARD VALIDATION WITH INTRA-FOLD OPTIMIZATION")
            print("="*80)
            print(f"Total observations: {len(valid_indices)}")
            print(f"Min training window: {self.config.min_train_window}")
            print(f"Validation window: {self.config.validation_window}")
            print(f"Step size: {self.config.step_size}")
            print(f"Parameter combinations per fold: {num_trials}")
            print("-"*80)
        
        while train_end_idx + self.config.validation_window <= len(valid_indices):
            fold_num += 1
            
            # Define windows
            train_indices = valid_indices[:train_end_idx]
            val_indices = valid_indices[train_end_idx:train_end_idx + self.config.validation_window]
            
            train_start_date = train_indices[0]
            train_end_date = train_indices[-1]
            val_start_date = val_indices[0]
            val_end_date = val_indices[-1]
            
            if verbose:
                print(f"\nFold {fold_num}:")
                print(f"  Train: {train_start_date.date()} to {train_end_date.date()} "
                      f"({len(train_indices)} days)")
                print(f"  Valid: {val_start_date.date()} to {val_end_date.date()} "
                      f"({len(val_indices)} days)")
            
            # -----------------------------------------------------------------
            # STEP 2a: OPTIMIZE on training data
            # -----------------------------------------------------------------
            train_df = df.loc[train_indices]
            
            optimal_params, is_metrics = self.optimizer.optimize(
                train_df, 
                verbose=verbose
            )
            
            param_history.append({
                'fold': fold_num,
                **optimal_params
            })
            
            is_sharpe = is_metrics.get('sharpe_ratio', 0)
            is_return = is_metrics.get('total_return', 0)
            is_max_dd = is_metrics.get('max_drawdown', 0)
            is_num_trades = is_metrics.get('num_trades', 0)
            
            # -----------------------------------------------------------------
            # STEP 2b: VALIDATE on out-of-sample data
            # -----------------------------------------------------------------
            val_df = df.loc[val_indices]
            
            # Create detector with optimal parameters
            optimal_detector = ThresholdRegimeDetector(
                self.config,
                vol_lookback=optimal_params['vol_lookback'],
                quantile_window=optimal_params['quantile_window'],
                regime_quantile=optimal_params['regime_quantile'],
                persistence_days=optimal_params['persistence_days']
            )
            
            # Detect regimes on validation data
            # Note: We use the full history up to validation end for regime detection
            # to maintain continuity
            full_val_df = df.loc[:val_end_date]
            df_with_regimes_val = optimal_detector.detect_regimes(full_val_df)
            
            # Extract validation period
            val_regimes = df_with_regimes_val.loc[val_indices]
            
            # Generate signals
            val_signals = val_regimes['regime'].apply(optimal_detector.get_signal)
            
            # Run backtest
            backtest_engine = BacktestEngine(self.config)
            val_results = backtest_engine.run_backtest(val_regimes, val_signals)
            
            # Calculate OOS metrics
            val_returns = val_results['strategy_return'].dropna()
            oos_num_trades = val_results['is_execution_day'].sum()
            oos_signal_changes = val_results['signal_change'].sum()
            
            if len(val_returns) > 10:
                oos_metrics = self.analytics.calculate_basic_metrics(val_returns)
                oos_sharpe = oos_metrics.get('sharpe_ratio', 0)
                oos_return = oos_metrics.get('total_return', 0)
                oos_max_dd = oos_metrics.get('max_drawdown', 0)
            else:
                oos_sharpe = 0
                oos_return = 0
                oos_max_dd = 0
            
            # Store OOS returns
            oos_returns_list.append(val_returns)
            
            # -----------------------------------------------------------------
            # STEP 2c: Calculate fold-level metrics
            # -----------------------------------------------------------------
            # Walk-Forward Efficiency
            if is_sharpe != 0 and not np.isnan(is_sharpe):
                wfe = oos_sharpe / is_sharpe
            else:
                wfe = 0
            
            fold_result = {
                'fold': fold_num,
                'train_start': train_start_date,
                'train_end': train_end_date,
                'val_start': val_start_date,
                'val_end': val_end_date,
                'train_days': len(train_indices),
                'val_days': len(val_indices),
                # In-sample metrics
                'is_return': is_return,
                'is_sharpe': is_sharpe,
                'is_max_dd': is_max_dd,
                'is_num_trades': is_num_trades,
                # Out-of-sample metrics
                'oos_return': oos_return,
                'oos_sharpe': oos_sharpe,
                'oos_max_dd': oos_max_dd,
                'oos_num_trades': oos_num_trades,
                'oos_signal_changes': oos_signal_changes,
                # Walk-forward efficiency
                'wfe': wfe,
                # Optimal parameters
                **{f'param_{k}': v for k, v in optimal_params.items()}
            }
            
            fold_results.append(fold_result)
            
            if verbose:
                print(f"  IS: Sharpe={is_sharpe:.3f}, Ret={is_return*100:.2f}%, "
                      f"Trades={is_num_trades}")
                print(f"  OOS: Sharpe={oos_sharpe:.3f}, Ret={oos_return*100:.2f}%, "
                      f"Trades={oos_num_trades}, WFE={wfe:.3f}")
            
            # Step forward
            train_end_idx += self.config.step_size
        
        # ---------------------------------------------------------------------
        # STEP 3: AGGREGATE METRICS
        # ---------------------------------------------------------------------
        if verbose:
            print("\n" + "-"*80)
            print("AGGREGATE WALK-FORWARD RESULTS")
            print("-"*80)
        
        # Combine all OOS returns
        if len(oos_returns_list) > 0:
            all_oos_returns = pd.concat(oos_returns_list)
        else:
            raise ValueError("No out-of-sample returns generated")
        
        # Calculate aggregate metrics
        agg_metrics = self.analytics.calculate_basic_metrics(all_oos_returns)
        
        # Average WFE across folds
        valid_wfes = [f['wfe'] for f in fold_results 
                     if not np.isnan(f['wfe']) and abs(f['wfe']) < 10]
        avg_wfe = np.mean(valid_wfes) if len(valid_wfes) > 0 else 0
        
        # Calculate skewness and kurtosis for DSR/PSR
        skewness = all_oos_returns.skew()
        kurtosis = all_oos_returns.kurtosis() + 3  # Convert to standard
        
        # DSR
        dsr = self.analytics.calculate_deflated_sharpe_ratio(
            sharpe=agg_metrics.get('sharpe_ratio', 0),
            num_observations=len(all_oos_returns),
            num_trials=num_trials * len(fold_results),  # Total trials
            skewness=skewness,
            kurtosis=kurtosis
        )
        
        # PSR
        psr = self.analytics.calculate_probabilistic_sharpe_ratio(
            sharpe=agg_metrics.get('sharpe_ratio', 0),
            num_observations=len(all_oos_returns),
            benchmark_sharpe=self.config.psr_benchmark_sharpe,
            skewness=skewness,
            kurtosis=kurtosis
        )
        
        # Hurst exponent
        hurst = self.analytics.calculate_hurst_exponent(all_oos_returns)
        
        # CDaR
        cdar = self.analytics.calculate_cdar(all_oos_returns)
        
        # PSI (using fold-level Sharpes)
        oos_sharpes = [f['oos_sharpe'] for f in fold_results]
        psi = self.analytics.calculate_parameter_stability_index(oos_sharpes)
        
        # GRI
        gri_components = {
            'dsr': dsr,
            'psr': psr,
            'wfe': avg_wfe,
            'hurst': hurst,
            'cdar': cdar,
            'psi': psi
        }
        gri = self.analytics.calculate_global_robustness_index(gri_components)
        
        aggregate_results = {
            'num_folds': len(fold_results),
            'total_oos_days': len(all_oos_returns),
            'total_param_trials': num_trials * len(fold_results),
            'oos_total_return': agg_metrics.get('total_return', 0),
            'oos_annual_return': agg_metrics.get('annual_return', 0),
            'oos_volatility': agg_metrics.get('volatility', 0),
            'oos_sharpe': agg_metrics.get('sharpe_ratio', 0),
            'oos_sortino': agg_metrics.get('sortino_ratio', 0),
            'oos_max_drawdown': agg_metrics.get('max_drawdown', 0),
            'oos_calmar': agg_metrics.get('calmar_ratio', 0),
            'avg_wfe': avg_wfe,
            'dsr': dsr,
            'psr': psr,
            'hurst': hurst,
            'cdar': cdar,
            'psi': psi,
            'gri': gri,
            'is_robust': gri > self.config.gri_threshold
        }
        
        if verbose:
            print(f"\nTotal Folds: {aggregate_results['num_folds']}")
            print(f"Total Parameter Trials: {aggregate_results['total_param_trials']}")
            print(f"OOS Total Return: {aggregate_results['oos_total_return']*100:.2f}%")
            print(f"OOS Annual Return: {aggregate_results['oos_annual_return']*100:.2f}%")
            print(f"OOS Sharpe: {aggregate_results['oos_sharpe']:.3f}")
            print(f"OOS Max DD: {aggregate_results['oos_max_drawdown']*100:.2f}%")
            print(f"\nRobustness Metrics:")
            print(f"  Avg WFE: {avg_wfe:.3f}")
            print(f"  DSR: {dsr:.3f}")
            print(f"  PSR: {psr:.3f}")
            print(f"  Hurst: {hurst:.3f}")
            print(f"  CDaR: {cdar*100:.2f}%")
            print(f"  PSI: {psi:.3f}")
            print(f"  GRI: {gri:.3f}")
            print(f"\nDeployment Ready: "
                  f"{'YES' if aggregate_results['is_robust'] else 'NO'} "
                  f"(GRI {'>' if aggregate_results['is_robust'] else '<='} "
                  f"{self.config.gri_threshold})")
        
        # ---------------------------------------------------------------------
        # STEP 4: Zero-Sharpe Fold Diagnostics
        # ---------------------------------------------------------------------
        zero_sharpe_folds = [f for f in fold_results if abs(f['oos_sharpe']) < 0.01]
        
        if len(zero_sharpe_folds) > 0 and verbose:
            print(f"\n[DIAGNOSTIC] {len(zero_sharpe_folds)} folds with zero OOS Sharpe:")
            for f in zero_sharpe_folds:
                print(f"  Fold {f['fold']}: {f['val_start'].date()} to {f['val_end'].date()}, "
                      f"trades={f['oos_num_trades']}, signals={f['oos_signal_changes']}")
        
        return {
            'fold_results': fold_results,
            'aggregate_metrics': aggregate_results,
            'oos_returns': all_oos_returns,
            'param_history': param_history
        }


# ==============================================================================
# COMPREHENSIVE VISUALIZATION MODULE
# ==============================================================================
# Creates detailed plots for strategy analysis
#
# V2.0 IMPROVEMENTS:
# - Equity curve with regime overlay
# - Per-fold IS vs OOS Sharpe comparison
# - Regime visualization on price chart
# - Monthly returns heatmap
# - Parameter stability across folds
# - Rolling IC analysis
# ==============================================================================

class StrategyVisualizer:
    """
    Comprehensive visualization suite for strategy analysis.
    
    V2.0 IMPROVEMENTS:
    -----------------
    - REGIME OVERLAY: Visualizes regimes on price chart
    - PER-FOLD ANALYSIS: IS vs OOS comparison
    - PARAMETER STABILITY: Shows how parameters change across folds
    - MONTHLY HEATMAP: Calendar view of returns
    - ROLLING IC: Predictive power over time
    """
    
    def __init__(self, config: StrategyConfig):
        self.config = config
        plt.style.use('seaborn-v0_8-whitegrid')
        
    def plot_comprehensive_dashboard(
        self,
        backtest_results: pd.DataFrame,
        regime_df: pd.DataFrame,
        wf_results: Dict[str, Any],
        save_path: Optional[str] = None
    ):
        """
        Create comprehensive multi-panel dashboard.
        """
        fig = plt.figure(figsize=(20, 28))
        gs = GridSpec(7, 2, figure=fig, hspace=0.35, wspace=0.25)
        
        # =====================================================================
        # Panel 1: Equity Curve with Benchmark (top, full width)
        # =====================================================================
        ax1 = fig.add_subplot(gs[0, :])
        
        portfolio_value = backtest_results['portfolio_value']
        benchmark_cumulative = (1 + backtest_results['benchmark_return']).cumprod() * self.config.initial_capital
        
        ax1.plot(portfolio_value.index, portfolio_value.values, 
                color=self.config.color_equity_curve, linewidth=1.5, label='Strategy')
        ax1.plot(benchmark_cumulative.index, benchmark_cumulative.values, 
                color=self.config.color_benchmark, linewidth=1.5, linestyle='--', 
                label=f'Benchmark ({self.config.benchmark_ticker})')
        
        ax1.set_ylabel('Portfolio Value ($)', fontsize=11)
        ax1.set_title('Threshold Regime Strategy V2.0 - Equity Curve', fontsize=14, fontweight='bold')
        ax1.legend(loc='upper left', fontsize=10)
        ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
        plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)
        ax1.grid(True, alpha=0.3)
        
        # =====================================================================
        # Panel 2: Price with Regime Overlay (NEW in V2.0)
        # =====================================================================
        ax2 = fig.add_subplot(gs[1, :])
        
        # Plot price
        price = regime_df['Close'].loc[backtest_results.index]
        ax2.plot(price.index, price.values, color='black', linewidth=1, label='Price')
        
        # Add regime background colors
        regime = backtest_results['regime']
        for i in range(len(regime) - 1):
            if pd.isna(regime.iloc[i]):
                continue
            
            start_date = regime.index[i]
            end_date = regime.index[i + 1]
            
            if regime.iloc[i] == 0:  # Low vol - invested
                color = self.config.color_regime_low_vol
                alpha = 0.2
            else:  # High vol - cash
                color = self.config.color_regime_high_vol
                alpha = 0.2
            
            ax2.axvspan(start_date, end_date, color=color, alpha=alpha)
        
        # Legend for regimes
        legend_elements = [
            Patch(facecolor=self.config.color_regime_low_vol, alpha=0.3, label='Low Vol (Invested)'),
            Patch(facecolor=self.config.color_regime_high_vol, alpha=0.3, label='High Vol (Cash)'),
            plt.Line2D([0], [0], color='black', linewidth=1, label='Price')
        ]
        ax2.legend(handles=legend_elements, loc='upper left', fontsize=9)
        ax2.set_ylabel('Price ($)', fontsize=11)
        ax2.set_title('Price with Regime Overlay', fontsize=12)
        ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)
        ax2.grid(True, alpha=0.3)
        
        # =====================================================================
        # Panel 3: Drawdown
        # =====================================================================
        ax3 = fig.add_subplot(gs[2, :])
        
        cumulative = portfolio_value / portfolio_value.iloc[0]
        running_max = cumulative.cummax()
        drawdown = (cumulative - running_max) / running_max
        
        ax3.fill_between(drawdown.index, drawdown.values * 100, 0, 
                        color=self.config.color_drawdown, alpha=0.5)
        ax3.plot(drawdown.index, drawdown.values * 100, 
                color=self.config.color_drawdown, linewidth=1)
        ax3.set_ylabel('Drawdown (%)', fontsize=11)
        ax3.set_title('Underwater Equity Curve', fontsize=12)
        ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45)
        ax3.grid(True, alpha=0.3)
        
        # =====================================================================
        # Panel 4: Walk-Forward IS vs OOS Sharpe
        # =====================================================================
        ax4 = fig.add_subplot(gs[3, 0])
        
        fold_results = wf_results['fold_results']
        folds = [f['fold'] for f in fold_results]
        is_sharpes = [f['is_sharpe'] for f in fold_results]
        oos_sharpes = [f['oos_sharpe'] for f in fold_results]
        
        x = np.arange(len(folds))
        width = 0.35
        
        ax4.bar(x - width/2, is_sharpes, width, label='In-Sample', 
               color=self.config.color_in_sample)
        ax4.bar(x + width/2, oos_sharpes, width, label='Out-of-Sample', 
               color=self.config.color_out_of_sample)
        
        ax4.set_xlabel('Fold', fontsize=10)
        ax4.set_ylabel('Sharpe Ratio', fontsize=10)
        ax4.set_title('Walk-Forward: IS vs OOS Sharpe', fontsize=12)
        ax4.set_xticks(x)
        ax4.set_xticklabels(folds, fontsize=8)
        ax4.legend(fontsize=9)
        ax4.axhline(y=0, color='black', linestyle='--', alpha=0.5)
        ax4.grid(True, alpha=0.3)
        
        # =====================================================================
        # Panel 5: Walk-Forward Efficiency by Fold
        # =====================================================================
        ax5 = fig.add_subplot(gs[3, 1])
        
        wfes = [min(max(f['wfe'], -2), 3) for f in fold_results]  # Clip for visibility
        colors = ['green' if w > 0 else 'red' for w in wfes]
        
        ax5.bar(folds, wfes, color=colors, alpha=0.7)
        ax5.axhline(y=self.config.min_wfe, color='orange', linestyle='--', 
                   label=f'Min WFE ({self.config.min_wfe})', linewidth=2)
        ax5.axhline(y=1.0, color='green', linestyle='--', 
                   label='Perfect WFE (1.0)', linewidth=2)
        
        ax5.set_xlabel('Fold', fontsize=10)
        ax5.set_ylabel('WFE (OOS/IS)', fontsize=10)
        ax5.set_title('Walk-Forward Efficiency by Fold', fontsize=12)
        ax5.legend(fontsize=9)
        ax5.grid(True, alpha=0.3)
        
        # =====================================================================
        # Panel 6: Regime Distribution
        # =====================================================================
        ax6 = fig.add_subplot(gs[4, 0])
        
        valid_regimes = backtest_results['regime'].dropna()
        regime_counts = valid_regimes.value_counts().sort_index()
        colors = [self.config.color_regime_low_vol, self.config.color_regime_high_vol]
        labels = ['Low Vol (Regime 0)', 'High Vol (Regime 1)']
        
        bars = ax6.bar(labels, regime_counts.values, color=colors, edgecolor='black')
        ax6.set_ylabel('Number of Days', fontsize=10)
        ax6.set_title('Regime Distribution', fontsize=12)
        
        for i, v in enumerate(regime_counts.values):
            ax6.text(i, v + len(valid_regimes)*0.01, 
                    f'{v}\n({v/len(valid_regimes)*100:.1f}%)', 
                    ha='center', fontsize=10)
        ax6.grid(True, alpha=0.3, axis='y')
        
        # =====================================================================
        # Panel 7: Returns by Regime (Box Plot)
        # =====================================================================
        ax7 = fig.add_subplot(gs[4, 1])
        
        merged = backtest_results.copy()
        regime_0_returns = merged[merged['regime'] == 0]['strategy_return'].dropna() * 100
        regime_1_returns = merged[merged['regime'] == 1]['strategy_return'].dropna() * 100
        
        bp = ax7.boxplot([regime_0_returns, regime_1_returns], 
                        positions=[1, 2], patch_artist=True, widths=0.6)
        
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        
        ax7.set_xticklabels(['Low Vol', 'High Vol'], fontsize=10)
        ax7.set_ylabel('Daily Return (%)', fontsize=10)
        ax7.set_title('Return Distribution by Regime', fontsize=12)
        ax7.axhline(y=0, color='black', linestyle='--', alpha=0.5)
        ax7.grid(True, alpha=0.3)
        
        # =====================================================================
        # Panel 8: Parameter Stability Across Folds (NEW in V2.0)
        # =====================================================================
        ax8 = fig.add_subplot(gs[5, 0])
        
        param_history = wf_results['param_history']
        folds_ph = [p['fold'] for p in param_history]
        vol_lb = [p['vol_lookback'] for p in param_history]
        q_win = [p['quantile_window'] / 10 for p in param_history]  # Scale for visibility
        pers = [p['persistence_days'] * 10 for p in param_history]  # Scale for visibility
        
        ax8.plot(folds_ph, vol_lb, 'o-', label='Vol Lookback', linewidth=2, markersize=4)
        ax8.plot(folds_ph, q_win, 's-', label='Quantile Win / 10', linewidth=2, markersize=4)
        ax8.plot(folds_ph, pers, '^-', label='Persistence * 10', linewidth=2, markersize=4)
        
        ax8.set_xlabel('Fold', fontsize=10)
        ax8.set_ylabel('Parameter Value', fontsize=10)
        ax8.set_title('Optimal Parameters Across Folds', fontsize=12)
        ax8.legend(fontsize=9)
        ax8.grid(True, alpha=0.3)
        
        # =====================================================================
        # Panel 9: Monthly Returns Heatmap
        # =====================================================================
        ax9 = fig.add_subplot(gs[5, 1])
        
        # Calculate monthly returns
        monthly_returns = backtest_results['strategy_return'].resample('ME').apply(
            lambda x: (1 + x).prod() - 1
        ) * 100
        
        # Create pivot table for heatmap
        monthly_df = pd.DataFrame({
            'Year': monthly_returns.index.year,
            'Month': monthly_returns.index.month,
            'Return': monthly_returns.values
        })
        
        pivot = monthly_df.pivot(index='Year', columns='Month', values='Return')
        
        # Plot heatmap
        sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
                   ax=ax9, cbar_kws={'label': 'Return (%)'}, annot_kws={'fontsize': 8})
        ax9.set_xlabel('Month', fontsize=10)
        ax9.set_ylabel('Year', fontsize=10)
        ax9.set_title('Monthly Returns Heatmap (%)', fontsize=12)
        
        # =====================================================================
        # Panel 10: Robustness Metrics Radar
        # =====================================================================
        ax10 = fig.add_subplot(gs[6, 0], projection='polar')
        
        agg_metrics = wf_results['aggregate_metrics']
        
        metrics_radar = ['DSR', 'PSR', 'WFE', 'Hurst', 'CDaR*', 'PSI']
        values_radar = [
            agg_metrics['dsr'],
            agg_metrics['psr'],
            min(1, max(0, agg_metrics['avg_wfe'])),
            agg_metrics['hurst'],
            max(0, min(1, 1 + agg_metrics['cdar'] * 2)),
            agg_metrics['psi']
        ]
        
        # Close the polygon
        angles = np.linspace(0, 2 * np.pi, len(metrics_radar), endpoint=False).tolist()
        values_radar_closed = values_radar + values_radar[:1]
        angles_closed = angles + angles[:1]
        
        ax10.plot(angles_closed, values_radar_closed, 'o-', linewidth=2, color='darkblue')
        ax10.fill(angles_closed, values_radar_closed, alpha=0.25, color='blue')
        ax10.set_xticks(angles)
        ax10.set_xticklabels(metrics_radar, fontsize=10)
        ax10.set_ylim(0, 1)
        ax10.set_title('Robustness Metrics', fontsize=12, pad=20)
        
        # Add threshold circle
        threshold_values = [self.config.gri_threshold] * len(angles_closed)
        ax10.plot(angles_closed, threshold_values, '--', color='red', alpha=0.5, 
                 label=f'GRI Threshold ({self.config.gri_threshold})')
        
        # =====================================================================
        # Panel 11: Performance Summary Table
        # =====================================================================
        ax11 = fig.add_subplot(gs[6, 1])
        ax11.axis('off')
        
        # Calculate final metrics
        final_return = (portfolio_value.iloc[-1] / self.config.initial_capital - 1) * 100
        benchmark_return = (benchmark_cumulative.iloc[-1] / self.config.initial_capital - 1) * 100
        
        strategy_returns = backtest_results['strategy_return'].dropna()
        annual_vol = strategy_returns.std() * np.sqrt(252) * 100
        sharpe = (strategy_returns.mean() * 252) / (strategy_returns.std() * np.sqrt(252)) if strategy_returns.std() > 0 else 0
        
        max_dd = drawdown.min() * 100
        num_trades = backtest_results['is_execution_day'].sum()
        total_costs = backtest_results['transaction_cost'].sum()
        
        summary_text = (
            f"PERFORMANCE SUMMARY\n"
            f"{'='*45}\n\n"
            f"Period: {backtest_results.index[0].date()} to {backtest_results.index[-1].date()}\n"
            f"Trading Days: {len(backtest_results)}\n\n"
            f"Strategy Return: {final_return:.2f}%\n"
            f"Benchmark Return: {benchmark_return:.2f}%\n"
            f"Excess Return: {final_return - benchmark_return:.2f}%\n\n"
            f"Annual Volatility: {annual_vol:.2f}%\n"
            f"Sharpe Ratio: {sharpe:.3f}\n"
            f"Max Drawdown: {max_dd:.2f}%\n\n"
            f"Number of Trades: {num_trades}\n"
            f"Total Transaction Costs: ${total_costs:,.2f}\n"
            f"Cost as % of Capital: {total_costs/self.config.initial_capital*100:.3f}%\n\n"
            f"{'='*45}\n"
            f"ROBUSTNESS ASSESSMENT\n"
            f"{'='*45}\n\n"
            f"GRI: {agg_metrics['gri']:.3f} (threshold: {self.config.gri_threshold})\n"
            f"DSR: {agg_metrics['dsr']:.3f}\n"
            f"PSR: {agg_metrics['psr']:.3f}\n"
            f"Avg WFE: {agg_metrics['avg_wfe']:.3f}\n"
            f"Hurst: {agg_metrics['hurst']:.3f}\n"
            f"CDaR: {agg_metrics['cdar']*100:.2f}%\n"
            f"PSI: {agg_metrics['psi']:.3f}\n\n"
            f"Deployment Ready: {'YES' if agg_metrics['is_robust'] else 'NO'}"
        )
        
        ax11.text(0.05, 0.95, summary_text, transform=ax11.transAxes, 
                fontsize=10, verticalalignment='top', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        # Save or show
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=self.config.dpi, bbox_inches='tight')
            print(f"\n[PLOT] Saved dashboard to: {save_path}")
        
        plt.close()
        
        return fig
    
    def plot_fold_diagnostics(
        self,
        wf_results: Dict[str, Any],
        save_path: Optional[str] = None
    ):
        """
        Create detailed fold-by-fold diagnostic plots.
        """
        fold_results = wf_results['fold_results']
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        folds = [f['fold'] for f in fold_results]
        
        # Plot 1: IS vs OOS Returns
        ax1 = axes[0, 0]
        is_returns = [f['is_return'] * 100 for f in fold_results]
        oos_returns = [f['oos_return'] * 100 for f in fold_results]
        
        x = np.arange(len(folds))
        width = 0.35
        
        ax1.bar(x - width/2, is_returns, width, label='IS Return', color='steelblue')
        ax1.bar(x + width/2, oos_returns, width, label='OOS Return', color='coral')
        ax1.set_xlabel('Fold')
        ax1.set_ylabel('Return (%)')
        ax1.set_title('In-Sample vs Out-of-Sample Returns')
        ax1.legend()
        ax1.axhline(y=0, color='black', linestyle='--', alpha=0.5)
        
        # Plot 2: Number of Trades per Fold
        ax2 = axes[0, 1]
        is_trades = [f['is_num_trades'] for f in fold_results]
        oos_trades = [f['oos_num_trades'] for f in fold_results]
        
        ax2.bar(x - width/2, is_trades, width, label='IS Trades', color='steelblue')
        ax2.bar(x + width/2, oos_trades, width, label='OOS Trades', color='coral')
        ax2.set_xlabel('Fold')
        ax2.set_ylabel('Number of Trades')
        ax2.set_title('Trade Frequency by Fold')
        ax2.legend()
        
        # Plot 3: Max Drawdown by Fold
        ax3 = axes[1, 0]
        is_dd = [f['is_max_dd'] * 100 for f in fold_results]
        oos_dd = [f['oos_max_dd'] * 100 for f in fold_results]
        
        ax3.bar(x - width/2, is_dd, width, label='IS Max DD', color='steelblue')
        ax3.bar(x + width/2, oos_dd, width, label='OOS Max DD', color='coral')
        ax3.set_xlabel('Fold')
        ax3.set_ylabel('Max Drawdown (%)')
        ax3.set_title('Maximum Drawdown by Fold')
        ax3.legend()
        
        # Plot 4: Scatter plot IS vs OOS Sharpe
        ax4 = axes[1, 1]
        is_sharpes = [f['is_sharpe'] for f in fold_results]
        oos_sharpes = [f['oos_sharpe'] for f in fold_results]
        
        ax4.scatter(is_sharpes, oos_sharpes, s=100, alpha=0.7, c=folds, cmap='viridis')
        
        # Add 45-degree line
        min_val = min(min(is_sharpes), min(oos_sharpes))
        max_val = max(max(is_sharpes), max(oos_sharpes))
        ax4.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, label='45° line')
        
        # Add fold labels
        for i, (x_val, y_val) in enumerate(zip(is_sharpes, oos_sharpes)):
            ax4.annotate(str(folds[i]), (x_val, y_val), fontsize=8, ha='center', va='bottom')
        
        ax4.set_xlabel('In-Sample Sharpe')
        ax4.set_ylabel('Out-of-Sample Sharpe')
        ax4.set_title('IS vs OOS Sharpe by Fold')
        ax4.legend()
        ax4.axhline(y=0, color='gray', linestyle='--', alpha=0.3)
        ax4.axvline(x=0, color='gray', linestyle='--', alpha=0.3)
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=self.config.dpi, bbox_inches='tight')
            print(f"[PLOT] Saved fold diagnostics to: {save_path}")
        
        plt.close()


# ==============================================================================
# MAIN EXECUTION
# ==============================================================================

def main():
    """
    Main execution function for Threshold Regime Trading Strategy V2.0.
    
    EXECUTION FLOW:
    1. Initialize configuration
    2. Download data from Yahoo Finance
    3. Run walk-forward validation with intra-fold optimization
    4. Execute full backtest with optimized parameters
    5. Generate comprehensive reports and visualizations
    """
    
    Path("results/volatility-threshold-regimes").mkdir(parents=True, exist_ok=True)

    print("="*80)
    print(" "*15 + "THRESHOLD REGIME TRADING STRATEGY V2.1")
    print(" "*10 + "Walk-Forward Validation with Signal-Change-Only Execution")
    print("="*80)
    
    # =========================================================================
    # STEP 1: CONFIGURATION
    # =========================================================================
    print("\n[STEP 1] Initializing Configuration...")
    
    config = StrategyConfig()
    
    print(f"  Asset: {config.ticker}")
    print(f"  Benchmark: {config.benchmark_ticker}")
    print(f"  Period: {config.start_date} to {config.end_date}")
    print(f"  Initial Capital: ${config.initial_capital:,.0f}")
    print(f"  Compounding Mode: {'Production' if config.compounding_mode else 'Research'}")
    print(f"  Persistence Filter: {config.persistence_days} days")
    print(f"  Vol Lookback Grid: {config.vol_lookback_grid}")
    print(f"  Quantile Window Grid: {config.quantile_window_grid}")
    print(f"  Optimization Objective: {config.optimization_objective}")
    
    # =========================================================================
    # STEP 2: DATA ACQUISITION
    # =========================================================================
    print("\n[STEP 2] Downloading Market Data from Yahoo Finance...")
    
    # Calculate extended start date for warm-up
    start_dt = datetime.strptime(config.start_date, "%Y-%m-%d")
    extended_start = start_dt - timedelta(days=config.warmup_buffer_days)
    extended_start_str = extended_start.strftime("%Y-%m-%d")
    
    print(f"  Extended start for warm-up: {extended_start_str}")
    
    df = fetch_historical_data(config.ticker, extended_start_str, config.end_date)

    print(f"  Data range: {df.index.min().date()} to {df.index.max().date()}")
    print(f"  Total observations: {len(df)}")
    
    # =========================================================================
    # STEP 3: WALK-FORWARD VALIDATION WITH INTRA-FOLD OPTIMIZATION
    # =========================================================================
    print("\n[STEP 3] Running Walk-Forward Validation with Intra-Fold Optimization...")
    
    wf_validator = WalkForwardValidator(config)
    
    wf_results = wf_validator.run_walk_forward(df, verbose=True)
    
    # =========================================================================
    # STEP 4: FULL BACKTEST WITH BEST PARAMETERS
    # =========================================================================
    print("\n[STEP 4] Running Full Backtest with Optimal Parameters...")
    
    # Find most common optimal parameters across folds
    param_history = wf_results['param_history']
    
    # Use parameters from last fold as representative
    final_params = param_history[-1] if len(param_history) > 0 else {
        'vol_lookback': config.vol_lookback,
        'quantile_window': config.quantile_window,
        'regime_quantile': config.regime_quantile,
        'persistence_days': config.persistence_days
    }
    
    print(f"  Using parameters from final fold:")
    print(f"    vol_lookback: {final_params.get('vol_lookback', config.vol_lookback)}")
    print(f"    quantile_window: {final_params.get('quantile_window', config.quantile_window)}")
    print(f"    regime_quantile: {final_params.get('regime_quantile', config.regime_quantile)}")
    print(f"    persistence_days: {final_params.get('persistence_days', config.persistence_days)}")
    
    # Create detector with final parameters
    final_detector = ThresholdRegimeDetector(
        config,
        vol_lookback=final_params.get('vol_lookback', config.vol_lookback),
        quantile_window=final_params.get('quantile_window', config.quantile_window),
        regime_quantile=final_params.get('regime_quantile', config.regime_quantile),
        persistence_days=final_params.get('persistence_days', config.persistence_days)
    )
    
    # Detect regimes
    df_with_regimes = final_detector.detect_regimes(df)
    
    # Filter to backtest period
    backtest_start = pd.to_datetime(config.start_date)
    backtest_end = pd.to_datetime(config.end_date)
    
    # Generate signals
    signals = df_with_regimes['regime'].apply(final_detector.get_signal)
    
    # Run full backtest
    backtest_engine = BacktestEngine(config)
    backtest_results = backtest_engine.run_backtest(df_with_regimes, signals)
    
    # Filter to backtest period for final results
    backtest_results = backtest_results[(backtest_results.index >= backtest_start) & 
                                        (backtest_results.index <= backtest_end)]
    
    # Calculate final metrics
    final_value = backtest_results['portfolio_value'].iloc[-1]
    total_return = (final_value / config.initial_capital - 1) * 100
    
    total_costs = backtest_results['transaction_cost'].sum()
    num_trades = backtest_results['is_execution_day'].sum()
    
    print(f"\n  Final Portfolio Value: ${final_value:,.2f}")
    print(f"  Total Return: {total_return:.2f}%")
    print(f"  Number of Trades: {num_trades}")
    print(f"  Total Transaction Costs: ${total_costs:,.2f}")
    print(f"  Cost as % of Initial Capital: {total_costs/config.initial_capital*100:.3f}%")
    
    # =========================================================================
    # STEP 5: VISUALIZATION
    # =========================================================================
    if config.enable_plots:
        print("\n[STEP 5] Generating Visualizations...")
        
        visualizer = StrategyVisualizer(config)
        
        # Main dashboard
        save_path = 'results/volatility-threshold-regimes/threshold_regime_v2_dashboard.png'
        visualizer.plot_comprehensive_dashboard(
            backtest_results=backtest_results,
            regime_df=df_with_regimes,
            wf_results=wf_results,
            save_path=save_path
        )
        
        # Fold diagnostics
        fold_path = 'results/volatility-threshold-regimes/threshold_regime_v2_fold_diagnostics.png'
        visualizer.plot_fold_diagnostics(
            wf_results=wf_results,
            save_path=fold_path
        )
    
    # =========================================================================
    # STEP 6: EXPORT RESULTS
    # =========================================================================
    print("\n[STEP 6] Exporting Results...")
    
    # Export backtest results
    backtest_results.to_csv('results/volatility-threshold-regimes/threshold_regime_v2_backtest.csv')
    print("  Saved: threshold_regime_v2_backtest.csv")
    
    # Export walk-forward results
    wf_df = pd.DataFrame(wf_results['fold_results'])
    wf_df.to_csv('results/volatility-threshold-regimes/threshold_regime_v2_walkforward.csv', index=False)
    print("  Saved: threshold_regime_v2_walkforward.csv")
    
    # Export parameter history
    param_df = pd.DataFrame(wf_results['param_history'])
    param_df.to_csv('results/volatility-threshold-regimes/threshold_regime_v2_params.csv', index=False)
    print("  Saved: threshold_regime_v2_params.csv")
    
    # Export regime data
    df_with_regimes.to_csv('results/volatility-threshold-regimes/threshold_regime_v2_data.csv')
    print("  Saved: threshold_regime_v2_data.csv")
    
    # Export aggregate metrics as JSON
    agg_metrics = wf_results['aggregate_metrics']
    with open('results/volatility-threshold-regimes/threshold_regime_v2_metrics.json', 'w') as f:
        json.dump(agg_metrics, f, indent=2, default=str)
    print("  Saved: threshold_regime_v2_metrics.json")
    
    # =========================================================================
    # FINAL SUMMARY
    # =========================================================================
    print("\n" + "="*80)
    print("EXECUTION COMPLETE - V2.1")
    print("="*80)
    
    agg = wf_results['aggregate_metrics']
    
    print(f"\nStrategy: Threshold Regime Trading V2.1 on {config.ticker}")
    print(f"Period: {backtest_results.index.min().date()} to {backtest_results.index.max().date()}")
    print(f"\nKey Results:")
    print(f"  Total Return: {total_return:.2f}%")
    print(f"  OOS Sharpe (Walk-Forward): {agg['oos_sharpe']:.3f}")
    print(f"  Number of Trades: {num_trades}")
    print(f"  Global Robustness Index: {agg['gri']:.3f}")
    print(f"\nV2.1 Critical Fix Applied:")
    print(f"  - Signal-change-only execution: YES (no daily rebalancing)")
    print(f"  - Intra-fold parameter optimization: YES")
    print(f"  - Signal persistence filter: {config.persistence_days} days")
    print(f"  - Fixed DSR calculation: DSR = {agg['dsr']:.3f}")
    print(f"  - Parameter combinations tested: {agg['total_param_trials']}")
    print(f"\nDeployment Recommendation: {'PROCEED' if agg['is_robust'] else 'REQUIRES FURTHER ANALYSIS'}")
    
    if not agg['is_robust']:
        print(f"\n  GRI ({agg['gri']:.3f}) is below threshold ({config.gri_threshold})")
        print("  Consider:")
        print("    - Adjusting parameter search space")
        print("    - Testing alternative regime detection methods")
        print("    - Extending training window")
        print("    - Adding additional signal filters")
    
    return {
        'config': config,
        'data': df_with_regimes,
        'backtest_results': backtest_results,
        'wf_results': wf_results
    }


if __name__ == "__main__":
    results = main()